
# Predicción de fumadores — Entrega final

## Objetivo

Utilizar el modelo Random Forest entrenado para predecir la variable `smoking` en los 5.692 registros sin etiqueta.

Se carga el pipeline completo, que incluye el preprocesamiento y el clasificador, y se aplica el umbral de decisión de 0.41 seleccionado durante la validación.

El resultado se exportará a un archivo CSV, conservando el identificador original de cada registro.


In [5]:

from pathlib import Path
import pandas as pd
import joblib
import json

# Rutas
DATA_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

# Cargar datos sin etiqueta
df_predict = pd.read_csv(DATA_DIR / "predict_clean.csv")

# Cargar modelo definitivo
modelo = joblib.load(MODELS_DIR / "smoking_model.joblib")

# Cargar configuración
with open(MODELS_DIR / "model_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Registros para predecir: {len(df_predict):,}")
print(f"Variables esperadas: {len(metadata['variables'])}")
print(f"Umbral de clasificación: {metadata['umbral']}")
print(f"ID disponible: {'ID' in df_predict.columns}")


Registros para predecir: 5,692
Variables esperadas: 25
Umbral de clasificación: 0.41
ID disponible: True


In [6]:

# Seleccionar las mismas variables utilizadas en entrenamiento
X_predict = df_predict[metadata["variables"]]

# Generar probabilidades y aplicar el umbral definitivo
probabilidades = modelo.predict_proba(X_predict)[:, 1]
predicciones = (probabilidades >= metadata["umbral"]).astype(int)

# Conservar el ID original y agregar la predicción
resultado = pd.DataFrame({
    "ID": df_predict["ID"],
    "smoking_prediction": predicciones
})

# Guardar archivo de entrega
OUTPUT_DIR = Path("../data/processed")
archivo_salida = OUTPUT_DIR / "smoking_predictions.csv"

resultado.to_csv(archivo_salida, index=False)

print(f"Predicciones generadas: {len(resultado):,}")
print(f"Fumadores predichos: {sum(predicciones):,}")
print(f"No fumadores predichos: {len(predicciones) - sum(predicciones):,}")
print(f"Valores válidos: {set(predicciones).issubset({0, 1})}")
print(f"Archivo guardado: {archivo_salida}")

display(resultado.head())


Predicciones generadas: 5,692
Fumadores predichos: 2,260
No fumadores predichos: 3,432
Valores válidos: True
Archivo guardado: ..\data\processed\smoking_predictions.csv


,ID,smoking_prediction
0,27358,1
1,27364,1
2,27368,1
3,27378,0
4,27381,0



## Insights y conclusiones del proyecto

### Insight 1. La calidad de los datos condiciona el entrenamiento

**Evidencia:** Notebook 01 — celdas 4, 5 y 6.

Durante el análisis exploratorio se detectó que Excel había representado numerosas mediciones numéricas mediante fracciones de día.

Se identificaron 18 columnas afectadas y se aplicó una reconstrucción diferenciada para mediciones enteras y decimales. La transformación se realizó tanto sobre los 50.000 registros etiquetados como sobre los 5.692 registros sin etiqueta.

**Interpretación:** Antes de entrenar un modelo, es fundamental comprobar que los valores representen correctamente las mediciones originales. La reconstrucción se fundamentó en los patrones observados y en la plausibilidad de los valores obtenidos, aunque no pudo contrastarse con una fuente original independiente.

### Insight 2. Existen diferencias clínicas entre fumadores y no fumadores

**Evidencia:** Notebook 01 — celda 8 (medianas por grupo) y celda 9 (boxplots).

Se observaron las siguientes medianas:

| Variable | No fumadores | Fumadores |
|---|---:|---:|
| Gtp | 21 | 37 |
| Triglicéridos | 96 | 132 |
| HDL | 58 | 52 |
| Hemoglobina | 14.2 | 15.5 |

Gtp presentó una mediana aproximadamente 76.2 % superior entre fumadores, mientras que los triglicéridos mostraron una diferencia del 37.5 %.

**Interpretación:** Estos indicadores presentan asociaciones con la variable objetivo y podrían aportar información predictiva. Sin embargo, las distribuciones se superponen y las diferencias no demuestran causalidad.

### Insight 3. Las variables clínicas mejoran el rendimiento predictivo

**Evidencia:** Notebook 02 — celda 9 (comparación entre Random Forest demográfico y completo).

| Variables utilizadas | F1-score en validación |
|---|---:|
| Demográficas | 0.7033 |
| Demográficas y clínicas | 0.7368 |

La incorporación de variables clínicas produjo una mejora absoluta de 0.0335 puntos de F1-score, equivalente a aproximadamente un 4.8 % relativo.

**Interpretación:** La comparación, manteniendo el algoritmo y optimizando los umbrales de ambos modelos sobre validación, respalda la hipótesis exploratoria de que los indicadores clínicos contienen información predictiva adicional.

### Insight 4. El algoritmo y el umbral influyen en el rendimiento

**Evidencia:** Notebook 02 — celdas 5 y 6 (regresión logística y umbral), celda 8 (Random Forest).

La regresión logística completa obtuvo un F1-score de 0.6578 con el umbral predeterminado de 0.50. Al ajustar el umbral a 0.39, el F1-score aumentó a 0.7076.

Random Forest, utilizando todas las variables y un umbral de 0.41, alcanzó un F1-score de 0.7368 en validación.

**Interpretación:** La selección del algoritmo y del umbral de decisión puede modificar sustancialmente el rendimiento. Los resultados sugieren que Random Forest aprovecha mejor las relaciones presentes en este conjunto de datos.

### Insight 5. El modelo detecta una elevada proporción de fumadores, pero genera falsos positivos

**Evidencia:** Notebook 02 — celda 10 (classification report y matriz de confusión).

En el conjunto de prueba independiente de 10.000 registros, Random Forest obtuvo:

| Métrica | Resultado |
|---|---:|
| F1-score (clase 1) | 0.7415 |
| Precision (clase 1) | 0.6407 |
| Recall (clase 1) | 0.8800 |
| Accuracy | 0.7751 |

La matriz de confusión mostró:

- Verdaderos negativos: 4.525
- Falsos positivos: 1.809
- Falsos negativos: 440
- Verdaderos positivos: 3.226

**Interpretación:** El modelo identificó correctamente al 88 % de los fumadores del conjunto de prueba. Sin embargo, clasificó incorrectamente como fumadoras a 1.809 personas no fumadoras. Esto evidencia un compromiso entre la capacidad de detección y la precisión de las predicciones positivas.

### Insight 6. El modelo puede aplicarse a registros sin etiqueta

**Evidencia:** Notebook 02 — celda 12 (entrenamiento y persistencia del modelo); Notebook 03 — celda 3 (generación de predicciones).

Una vez seleccionado y evaluado el algoritmo, se entrenó una versión definitiva con los 50.000 registros etiquetados.

Se aplicó el modelo a 5.692 registros sin etiqueta y se obtuvieron:

- 2.260 predicciones de fumadores.
- 3.432 predicciones de no fumadores.

Las predicciones se guardaron en `data/processed/smoking_predictions.csv`, conservando los identificadores originales.

**Interpretación:** El procedimiento permitió completar la tarea predictiva solicitada. Estas cantidades representan predicciones del modelo y no permiten conocer la distribución real de fumadores en los registros sin etiqueta.

## Conclusión general

Se desarrolló un sistema de clasificación binaria para predecir el hábito de fumar a partir de variables demográficas y clínicas.

El proceso incluyó análisis exploratorio, detección y reconstrucción de mediciones, comparación de algoritmos, optimización de umbrales y evaluación sobre un conjunto de prueba independiente.

Los experimentos respaldaron la hipótesis de que las variables clínicas aportan información predictiva adicional. Random Forest fue el mejor algoritmo evaluado y alcanzó un F1-score de 0.7415 en la prueba independiente.

Posteriormente, se entrenó el modelo definitivo con los 50.000 registros etiquetados y se generaron las 5.692 predicciones requeridas.

Entre las limitaciones se destacan la reconstrucción inferida de mediciones afectadas por Excel, la existencia de falsos positivos y la ausencia de validación externa.

Por lo tanto, el modelo constituye una solución experimental de aprendizaje automático, no una herramienta de diagnóstico clínico.

